# B2-024-gpu-scientific-ml-capstone — Practice p06 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** gpu-colab-l4-workflow

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

NESTED = {
    "images": torch.zeros(2, 1, 8, 8),
    "extras": [torch.arange(3), "shape-batch", 7],
    "pair": (torch.ones(2), torch.tensor([1, 2])),
    "name": "probe",
}
CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "steps": 5, "batch_size": 64, "simulated_rows": 0}

## Solution

**Design.**

* `get_device` is the one-line device decision of Session 1, made once.
* `move_batch` walks the structure recursively. Tensors are replaced by `tensor.to(device)` (which returns a new tensor and never touches the original); `tuple`/`list` are rebuilt with `type(batch)(...)` so the container type and order survive; `dict` is rebuilt with a comprehension over its items so the keys survive; everything else is returned as is. Because every container is rebuilt rather than mutated, the input structure is never modified.
* `train_step` keeps **only** the forward pass and the loss inside `torch.autocast` (and uses no context at all when `amp_dtype is None`, the CI path); `backward()` and `step()` run outside the context. The returned loss is `loss.item()`, a detached Python float, so no graph is kept alive (Session 1's leak pitfall).
* `run(config)` is the single code path: it reads the device, `amp_dtype`, step count, batch size, and data source (stored rows, or `simulated_rows` fresh rows) from the dictionary, so only the dictionary changes between the CPU and the L4.

In [ ]:
from contextlib import nullcontext


def get_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


def move_batch(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device)
    if isinstance(batch, (tuple, list)):
        return type(batch)(move_batch(item, device) for item in batch)
    if isinstance(batch, dict):
        return {key: move_batch(value, device) for key, value in batch.items()}
    return batch


def train_step(model, batch, optimizer, device, amp_dtype):
    images, labels = move_batch(batch, device)
    optimizer.zero_grad(set_to_none=True)
    context = nullcontext() if amp_dtype is None else torch.autocast(device_type=device.type, dtype=amp_dtype)
    with context:
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
    loss.backward()
    optimizer.step()
    return {"loss": loss.item(), "logits_dtype": logits.dtype}


images, labels = capstone_data.train_rows("shapes_supervised")
FIXED_BATCH = (images[:64], labels[:64])


def make_model_and_optimizer(device):
    torch.manual_seed(SEED)
    model = capstone_data.ShapeCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    return model, optimizer


def run(config):
    device = torch.device(config["device"])
    model, optimizer = make_model_and_optimizer(device)
    if config["simulated_rows"] == 0:
        source_x, source_y = images, labels
    else:
        source_x, source_y = capstone_data.simulate("shapes_supervised", config["simulated_rows"], seed=1)
    batch = (source_x[: config["batch_size"]], source_y[: config["batch_size"]])
    return [train_step(model, batch, optimizer, device, config["amp_dtype"])["loss"] for _ in range(config["steps"])]

**Probes 1–3: device choice and batch moving.**
The `meta` device carries shapes and dtypes but no data, so it lets a CPU-only machine prove that *every* tensor in the nested structure was moved, while the originals stay on the CPU.

In [ ]:
device_probe = get_device()
nested_snapshot = {"images": NESTED["images"].clone(), "arange": NESTED["extras"][0].clone()}

cpu_moved = move_batch(NESTED, torch.device("cpu"))
meta_moved = move_batch(NESTED, torch.device("meta"))


def tensors_in(obj):
    if isinstance(obj, torch.Tensor):
        yield obj
    elif isinstance(obj, (tuple, list)):
        for item in obj:
            yield from tensors_in(item)
    elif isinstance(obj, dict):
        for value in obj.values():
            yield from tensors_in(value)


meta_tensors = list(tensors_in(meta_moved))
print("device:", device_probe, "| cpu-moved types:", {k: type(v).__name__ for k, v in cpu_moved.items()})
print("meta tensors:", [t.device.type for t in meta_tensors],
      "| originals:", [t.device.type for t in tensors_in(NESTED)])

**Probes 4–5: the training step with and without autocast.**
With `amp_dtype=None` the logits are `float32`; under CPU `bfloat16` autocast the convolutions and the linear head run in `bfloat16`, so the logits are `bfloat16`, while the parameters remain `float32` master weights and their gradients are `float32` (autocast casts copies, and gradients flow back to the `float32` leaves).

In [ ]:
device = torch.device("cpu")
model, optimizer = make_model_and_optimizer(device)
fp32_results = [train_step(model, FIXED_BATCH, optimizer, device, None) for _ in range(5)]
fp32_losses = [r["loss"] for r in fp32_results]
print("float32 losses:", [round(l, 4) for l in fp32_losses])

model_bf, optimizer_bf = make_model_and_optimizer(device)
bf_result = train_step(model_bf, FIXED_BATCH, optimizer_bf, device, torch.bfloat16)
param_dtypes = {p.dtype for p in model_bf.parameters()}
grad_dtypes = {p.grad.dtype for p in model_bf.parameters()}
print("bf16 step:", bf_result, "| param dtypes:", param_dtypes, "| grad dtypes:", grad_dtypes)

**Probe 6: one config dict, one code path.**

In [ ]:
cpu_losses = run(CPU_CONFIG)
print("run(CPU_CONFIG):", [round(l, 4) for l in cpu_losses])

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

Run the same `run(config)` with a replacement config dict on a Colab L4 runtime. `L4_CONFIG` has exactly the keys of `CPU_CONFIG`; only the values change, and `train_step`, `move_batch`, and `run` are the code you tested on the CPU (the 2048-row batch comes from `simulated_rows`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "steps": 200, "batch_size": 2048, "simulated_rows": 2048}
losses = run(L4_CONFIG)
```

**What to observe.** `get_device()` reports `cuda`; `logits_dtype` is `torch.bfloat16`; time per step for `run({**L4_CONFIG, "amp_dtype": None})` versus `run(L4_CONFIG)` at batch 2048 (on the L4, `bfloat16` should be faster, unlike on the CPU); and `torch.cuda.max_memory_allocated()` after one step for both settings.

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
import math

# Probe 1
assert (get_device().type == "cuda") == torch.cuda.is_available()
# Probe 2
assert isinstance(cpu_moved, dict) and list(cpu_moved) == list(NESTED)
assert isinstance(cpu_moved["images"], torch.Tensor)
assert type(cpu_moved["extras"]) is list and type(cpu_moved["pair"]) is tuple and type(cpu_moved["name"]) is str
assert cpu_moved["name"] == "probe" and cpu_moved["extras"][1] == "shape-batch" and cpu_moved["extras"][2] == 7
assert torch.equal(cpu_moved["images"], NESTED["images"])
assert torch.equal(cpu_moved["extras"][0], NESTED["extras"][0])
assert torch.equal(cpu_moved["pair"][0], NESTED["pair"][0]) and torch.equal(cpu_moved["pair"][1], NESTED["pair"][1])
# Probe 3
assert len(meta_tensors) == 4 and all(t.device.type == "meta" for t in meta_tensors)
assert all(t.device.type == "cpu" for t in tensors_in(NESTED))
assert type(meta_moved["extras"]) is list and type(meta_moved["pair"]) is tuple and meta_moved["name"] == "probe"
assert torch.equal(NESTED["images"], nested_snapshot["images"]) and torch.equal(NESTED["extras"][0], nested_snapshot["arange"])
# Probe 4
assert all(r["logits_dtype"] == torch.float32 for r in fp32_results)
assert all(isinstance(l, float) and math.isfinite(l) for l in fp32_losses)
assert fp32_losses[4] < fp32_losses[0]
# Probe 5
assert bf_result["logits_dtype"] == torch.bfloat16 and math.isfinite(bf_result["loss"])
assert param_dtypes == {torch.float32} and grad_dtypes == {torch.float32}
# Probe 6
assert len(cpu_losses) == 5 and all(math.isfinite(l) for l in cpu_losses)
assert cpu_losses == fp32_losses          # same seed, same batch, same code path